# CP4 - Case iFood

Cognitive Data Science + Machine Learning & Modelling

Grupo Perceptron, sala 1TIAPZ-2026.

Execute as células nesta ordem. A senha Oracle nunca deve ser escrita no notebook.

In [ ]:
!pip -q install numpy==2.4.4 pandas==3.0.2 scikit-learn==1.9.1 oracledb==3.3.0
import getpass
import oracledb
import pandas as pd

## 1. Ler a tabela do Oracle

Antes desta célula, crie e carregue `IFOOD_CUSTOMERS` no SQL Developer. No Colab, abra o painel de chave `Secrets`, crie `ORACLE_PASSWORD`, informe a senha e ative o acesso para este notebook. Se o segredo não estiver configurado, a célula solicitará a senha sem exibi-la.

In [ ]:
ORACLE_USER = 'RM573854'
try:
    from google.colab import userdata
    ORACLE_PASSWORD = userdata.get('ORACLE_PASSWORD')
except Exception:
    ORACLE_PASSWORD = None
if not ORACLE_PASSWORD:
    ORACLE_PASSWORD = getpass.getpass('Senha Oracle (não exibida): ')
ORACLE_DSN = oracledb.makedsn('oracle.fiap.com.br', 1521, service_name='orcl')

SQL = '''SELECT
    ID, YEAR_BIRTH, EDUCATION, MARITAL_STATUS, INCOME, KIDHOME, TEENHOME,
    DT_CUSTOMER, RECENCY, MNTWINES, MNTFRUITS, MNTMEATPRODUCTS,
    MNTFISHPRODUCTS, MNTSWEETPRODUCTS, MNTGOLDPRODS, NUMDEALSPURCHASES,
    NUMWEBPURCHASES, NUMCATALOGPURCHASES, NUMSTOREPURCHASES,
    NUMWEBVISITSMONTH, ACCEPTEDCMP3, ACCEPTEDCMP4, ACCEPTEDCMP5,
    ACCEPTEDCMP1, ACCEPTEDCMP2, COMPLAIN, Z_COSTCONTACT, Z_REVENUE, RESPONSE
FROM IFOOD_CUSTOMERS
ORDER BY ID'''

with oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN) as connection:
    with connection.cursor() as cursor:
        cursor.execute(SQL)
        columns = [item[0] for item in cursor.description]
        df = pd.DataFrame(cursor.fetchall(), columns=columns)

print('shape:', df.shape)
print('INCOME nulo:', int(df['INCOME'].isna().sum()))

In [ ]:
"""Pipeline reproduzivel da Parte 2 do CP4.

Pode receber o DataFrame retornado pelo Oracle ou ler o CSV para teste local.
O holdout fica separado antes do tuning. Perfis de features repetidos ficam no
mesmo grupo entre treino, teste e folds. Imputacao e one-hot ficam dentro do
Pipeline, portanto usam apenas os dados de treino durante o ajuste. A ordenacao
por ID e o random_state fixo mantem o split reproduzivel no Oracle e no CSV.
"""

from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier


RANDOM_STATE = 42
DATA_PATH = Path("data.csv")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [str(column).upper() for column in result.columns]
    return result


def build_features(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    data = normalize_columns(df)
    if "ID" in data.columns:
        data = data.sort_values("ID", kind="mergesort").reset_index(drop=True)
    data["DT_CUSTOMER"] = pd.to_datetime(data["DT_CUSTOMER"], errors="coerce")

    # Features criadas antes do split, sem usar RESPONSE.
    data["AGE_AT_2014"] = 2014 - data["YEAR_BIRTH"]
    data["TOTAL_CHILDREN"] = data["KIDHOME"] + data["TEENHOME"]
    spend_columns = [
        "MNTWINES", "MNTFRUITS", "MNTMEATPRODUCTS",
        "MNTFISHPRODUCTS", "MNTSWEETPRODUCTS", "MNTGOLDPRODS",
    ]
    purchase_columns = [
        "NUMDEALSPURCHASES", "NUMWEBPURCHASES",
        "NUMCATALOGPURCHASES", "NUMSTOREPURCHASES",
    ]
    campaign_columns = [
        "ACCEPTEDCMP1", "ACCEPTEDCMP2", "ACCEPTEDCMP3",
        "ACCEPTEDCMP4", "ACCEPTEDCMP5",
    ]
    data["TOTAL_SPEND"] = data[spend_columns].sum(axis=1)
    data["TOTAL_PURCHASES"] = data[purchase_columns].sum(axis=1)
    data["CAMPAIGNS_ACCEPTED"] = data[campaign_columns].sum(axis=1)
    data["WEB_PURCHASE_SHARE"] = (
        data["NUMWEBPURCHASES"] / data["TOTAL_PURCHASES"].replace(0, 1)
    )
    data["CUSTOMER_TENURE_DAYS"] = (
        pd.Timestamp("2014-06-30") - data["DT_CUSTOMER"]
    ).dt.days

    y = data.pop("RESPONSE").astype(int)
    X = data.drop(
        columns=["ID", "YEAR_BIRTH", "DT_CUSTOMER", "Z_COSTCONTACT", "Z_REVENUE"]
    )
    groups = pd.util.hash_pandas_object(X, index=False)
    return X, y, groups


def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_columns = X.select_dtypes(include="number").columns.tolist()
    categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric, numeric_columns),
        ("categorical", categorical, categorical_columns),
    ])


def evaluate(name: str, model: Pipeline, X_test: pd.DataFrame, y_test: pd.Series) -> dict:
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    metrics = {
        "model": name,
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    }
    print(pd.Series(metrics).to_string())
    return metrics


def run(df: pd.DataFrame) -> pd.DataFrame:
    X, y, groups = build_features(df)
    holdout = StratifiedGroupKFold(
        n_splits=5, shuffle=True, random_state=RANDOM_STATE
    )
    train_index, test_index = next(holdout.split(X, y, groups))
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    groups_train = groups.iloc[train_index]
    groups_test = groups.iloc[test_index]
    if set(groups_train).intersection(groups_test):
        raise RuntimeError("Um perfil de features apareceu no treino e no teste.")
    print(
        f"holdout train={len(train_index)} test={len(test_index)} "
        f"positive_test={int(y_test.sum())} ({y_test.mean():.2%})"
    )

    reference = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])
    reference.fit(X_train, y_train)
    results = [evaluate("reference", reference, X_test, y_test)]

    tuned = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])
    search = GridSearchCV(
        tuned,
        param_grid={
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [3, 5, 8, None],
            "model__min_samples_leaf": [1, 5, 10, 20],
            "model__class_weight": [None, "balanced"],
        },
        scoring="roc_auc",
        cv=StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=RANDOM_STATE
        ),
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train, groups=groups_train)
    print("best_params")
    print(search.best_params_)
    results.append(evaluate("tuned", search.best_estimator_, X_test, y_test))
    return pd.DataFrame(results)




## 2. EDA

A análise verifica o desbalanceamento do alvo, valores ausentes, duplicidades e perfis de features repetidos. A taxa global de RESPONSE descreve o desbalanceamento; não comparamos o alvo por grupo antes do holdout.

In [ ]:
eda = df.copy()
eda.columns = [str(column).upper() for column in eda.columns]
source_columns = len(eda.columns)
X_eda, y_eda, groups_eda = build_features(eda)
profile_sizes = pd.Series(groups_eda).value_counts()
positive = int(y_eda.sum())
summary = pd.DataFrame({
    'Indicador': ['Linhas', 'Colunas da query', 'Resposta positiva', 'INCOME nulo', 'Linhas completas duplicadas', 'IDs duplicados', 'Perfis de features repetidos', 'Linhas excedentes nesses perfis'],
    'Resultado': [
        len(eda), source_columns, f'{positive} ({positive / len(eda):.2%})',
        int(eda['INCOME'].isna().sum()), int(eda.duplicated().sum()), int(eda['ID'].duplicated().sum()),
        int((profile_sizes > 1).sum()), int((profile_sizes - 1).clip(lower=0).sum())
    ],
})
display(summary)
campaign_distribution = (
    X_eda['CAMPAIGNS_ACCEPTED'].value_counts().sort_index()
    .rename_axis('CAMPAIGNS_ACCEPTED').reset_index(name='CLIENTES')
)
display(campaign_distribution)

## 3. Feature Engineering

AGE_AT_2014 aproxima a idade; TOTAL_CHILDREN resume a composição familiar; TOTAL_SPEND e TOTAL_PURCHASES medem valor e atividade de compra; CAMPAIGNS_ACCEPTED resume respostas a campanhas anteriores; WEB_PURCHASE_SHARE representa a preferência de canal; CUSTOMER_TENURE_DAYS mede o tempo de relacionamento. Nenhuma dessas variáveis usa RESPONSE. Perfis idênticos em X são agrupados para que nunca apareçam ao mesmo tempo no treino e no teste.

## 4. Referencia, preparacao e tuning\n\nA referencia reproduz a DecisionTreeClassifier sem tuning, com os parametros padrao do scikit-learn e random_state=42. Ela usa o mesmo holdout, pre-processamento e metricas que o modelo ajustado. O tuning avalia criterion, max_depth, min_samples_leaf e class_weight. O holdout usa o primeiro fold de StratifiedGroupKFold com cinco partes, preservando aproximadamente 80/20 e a proporcao de RESPONSE sem dividir perfis repetidos. A validacao cruzada tambem e por grupos e ocorre somente no treino. Imputacao e one-hot encoding ficam dentro do Pipeline.

In [ ]:
metrics = run(df)
metrics

## 5. Resultado

Compare precision, recall, F1 e ROC AUC no holdout reservado. Discuta os quatro indicadores; ROC AUC maior não implica que precision, recall e F1 também aumentem. Execute todas as células no Colab e salve o notebook com as saídas antes de publicar a versão executada.